# Ornith-1.5-35B-A3B — Download GGUF & Buat Kaggle Dataset

Notebook ini khusus untuk **mengunduh model GGUF dari Hugging Face** dan
**mengunggahnya sebagai Kaggle Dataset**, terpisah dari notebook yang menjalankan server.

Tidak perlu GPU untuk notebook ini — cukup CPU dan disk yang cukup (model bisa puluhan GB,
pastikan kuota disk Kaggle mencukupi untuk quant yang dipilih).


> ⚠️ **Soal disk space:** `/kaggle/working` dibatasi kuota kecil (~20GB, dipakai untuk output).
> Notebook ini mengunduh model ke `/kaggle/temp` (scratch disk yang lebih besar & tidak dihitung
> ke kuota output), lalu baru meng-upload-nya sebagai dataset dari lokasi tersebut.


In [1]:
# 1. Install tools
!pip install -q huggingface_hub kaggle


In [2]:
# 1b. Cek disk space yang tersedia
!df -h /kaggle/working /kaggle/temp 2>/dev/null || df -h


Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G   15G  5.0G  75% /kaggle/working
overlay         7.9T  6.9T 1007G  88% /


In [3]:
# 2. Download GGUF dari Hugging Face (ke /kaggle/temp, disk lebih lega)
from huggingface_hub import hf_hub_download
import os

# /kaggle/temp = scratch disk, TIDAK dihitung ke kuota output /kaggle/working
OUTPUT_DIR = "/kaggle/temp/ornith-gguf"
os.makedirs(OUTPUT_DIR, exist_ok=True)

REPO_ID = "ornith-ai/Ornith-1.5-35B-A3B-GGUF"
FILENAME = "Ornith-1.5-35B-Q4_K_M.gguf"  # ganti sesuai quant yang mau dipakai
                                          # opsi lain: Q5_K_M, Q6_K, Q8_0, BF16

path = hf_hub_download(
    repo_id=REPO_ID,
    filename=FILENAME,
    local_dir=OUTPUT_DIR,
    local_dir_use_symlinks=False,  # pastikan file fisik tersalin, bukan symlink ke cache
)
print("Downloaded to:", path)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Downloaded to: /kaggle/temp/ornith-gguf/Ornith-1.5-35B-Q4_K_M.gguf


In [8]:
# 3. Buat dataset-metadata.json
import json

KAGGLE_USERNAME = "CHANGE_KAGGLE_USERNAME"  # <-- ganti dengan username Kaggle kamu
DATASET_SLUG = "ornith-1-5-35b-a3b-q4km-gguf"

metadata = {
    "title": "Ornith 1.5 35B A3B Q4 K M GGUF",
    "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}]
}

with open(f"{OUTPUT_DIR}/dataset-metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print(metadata)


{'title': 'Ornith 1.5 35B A3B Q4 K M GGUF', 'id': 'baimnot/ornith-1-5-35b-a3b-q4km-gguf', 'licenses': [{'name': 'CC0-1.0'}]}


In [9]:
# 4. Setup Kaggle API credentials (berbasis Kaggle Secrets)
#
# Sebelum menjalankan cell ini:
#   1. Buka menu  Add-ons -> Secrets
#   2. Add Secret -> Label: kaggle_json -> Value: seluruh isi file kaggle.json kamu
#      (dari kaggle.com/settings -> Create New API Token, file yang ke-download)
#   3. PENTING: aktifkan toggle/checkbox secret tersebut untuk notebook INI
#      (secret harus di-attach per sesi, tidak otomatis walau sudah ada di akun kamu)
#   4. Baru jalankan cell ini

import os
from kaggle_secrets import UserSecretsClient

os.makedirs("/root/.kaggle", exist_ok=True)

secrets = UserSecretsClient()
try:
    kaggle_json = secrets.get_secret("kaggle_json")
except Exception as e:
    raise RuntimeError(
        "Secret 'kaggle_json' tidak ditemukan/tidak ter-attach ke sesi ini.\n"
        "Cek: Add-ons -> Secrets -> pastikan label PERSIS 'kaggle_json' dan "
        "toggle-nya ON untuk notebook ini, lalu jalankan ulang cell ini."
    ) from e

with open("/root/.kaggle/kaggle.json", "w") as f:
    f.write(kaggle_json)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle credentials berhasil dimuat dari Secrets.")


Kaggle credentials berhasil dimuat dari Secrets.


In [10]:
# 5. Push ke Kaggle Datasets (buat dataset baru)
!kaggle datasets create -p {OUTPUT_DIR} --dir-mode zip


Starting upload for file Ornith-1.5-35B-Q4_K_M.gguf
100%|██████████████████████████████████████| 20.2G/20.2G [13:45<00:00, 26.3MB/s]
Upload successful: Ornith-1.5-35B-Q4_K_M.gguf (20GB)
Starting upload for file .cache.zip
100%|██████████████████████████████████████████| 907/907 [00:00<00:00, 1.04kB/s]
Upload successful: .cache.zip (907B)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/baimnot/ornith-1-5-35b-a3b-q4km-gguf


In [7]:
# 5b. (Alternatif) Buat versi baru untuk dataset yang sudah ada
# !kaggle datasets version -p {OUTPUT_DIR} -m "update quant" --dir-mode zip


## Selesai

Setelah proses upload selesai, dataset akan tersedia di:

`https://www.kaggle.com/datasets/USERNAME_KAMU/ornith-1-5-35b-a3b-q4km-gguf`

Dataset ini bisa langsung di-**attach** (menu **Add Data**) ke notebook lain yang menjalankan
`llama-server` (mis. notebook `ornith_llama_server_cloudflare.ipynb`).
